In [1]:
# 1) Cargar modelo
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

# === Configura rutas ===
MODEL_ID = "models/qwen3-1p7b-qlora"
CKPT_DIR = "outputs/qwen3-1p7b-qlora/checkpoint-225"   # tu checkpoint entrenado

# === Carga tokenizer ===
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# === Carga modelo base en 4bit ===
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16
)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, device_map="auto", quantization_config=bnb_cfg, trust_remote_code=True
)

# === Monta el adaptador LoRA fine-tuneado ===
model = PeftModel.from_pretrained(base_model, CKPT_DIR)
model.eval()

print("✅ Modelo y adaptador cargados correctamente")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Modelo y adaptador cargados correctamente


C:\Users\jsoa\AppData\Roaming\Python\Python312\site-packages\peft\tuners\tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
# ============================
# Generar PLS con CoT Implicito Qwen (HF) usando columnas: name, article, summary
# ============================

import re
import time  # NEW
from pathlib import Path
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm

@torch.inference_mode()
def generate_pls_batch(
    data: str,
    prompt_fn,
    batch_size: int = 2,
    max_new_tokens: int = 380,     # suficiente para 4–6 oraciones
    temperature: float = 0.0,      # determinista → mejor factualidad
    top_p: float = 1.0,
    num_beams: int = 1,
    repetition_penalty: float = 1.02,
    no_repeat_ngram_size: int = 4,
):
    texts = data['article'].fillna("").astype(str).tolist()
    df_out = data.copy()

    # Calcula un input máximo seguro
    max_ctx = getattr(model.config, "max_position_embeddings", 4096)
    max_input_len = max(8, max_ctx - max_new_tokens)

    outputs = []
    latencies = []  # NEW
    pbar = tqdm(total=len(texts), desc='Generando resúmenes', unit="sample")

    try:
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            prompts = [prompt_fn(t) for t in batch_texts]

            enc = tokenizer(
                prompts,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=max_input_len,
            )
            input_ids = enc["input_ids"].to(model.device)
            attention_mask = enc["attention_mask"].to(model.device)

            # --- NEW: medir tiempo del batch con sync de GPU
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            t0 = time.perf_counter()

            gen_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
                do_sample=(temperature > 0.0 or top_p < 1.0),
                temperature=temperature,
                top_p=top_p,
                num_beams=num_beams,
                no_repeat_ngram_size=no_repeat_ngram_size,
                repetition_penalty=repetition_penalty,
                use_cache=True,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            )

            if torch.cuda.is_available():
                torch.cuda.synchronize()
            t1 = time.perf_counter()
            per_item_latency = (t1 - t0)  # NEW

            # Cortar por muestra usando la longitud REAL (no el ancho padded)
            batch_out = []
            input_lens = attention_mask.sum(dim=1)  # [B]
            for row in range(input_ids.size(0)):
                ilen = int(input_lens[row].item())
                gen_only = gen_ids[row, ilen:]        # ← clave: corta desde fin del prompt de ESA fila
                text = tokenizer.decode(gen_only, skip_special_tokens=True).strip()
                batch_out.append(text)

            outputs.extend(batch_out)
            latencies.extend([per_item_latency] * len(batch_texts))  # NEW
            pbar.update(len(batch_texts))
    finally:
        pbar.close()

    df_out['gen_summary'] = outputs
    df_out['latency_s'] = latencies  # NEW
    return df_out

# O el CoT factual corto
def prompt_cot_factual(t):
    return f"""You are a helpful medical writer.
            Think briefly before answering:
            - Use only statements explicitly present in the source.
            - Keep names and numbers exactly as written.
            - 4–6 sentences, ≤120 words. Do not show your reasoning.

            Scientific text:
            {t}

            Plain summary:"""

DATA_DIR = Path("data-sources/pre-processed")
RESULTS_DIR = Path("models/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

df_test = pd.read_csv(DATA_DIR / "data_finetuning_test.csv")
df_output = generate_pls_batch(
    data=df_test,
    prompt_fn=prompt_cot_factual,   # o prompt_sencillo
    batch_size=5,
    max_new_tokens=380,
    temperature=0.0,
    top_p=1.0,
)

csv_out = RESULTS_DIR / "summaries_qwen3_COT.csv"
df_output.to_csv(csv_out, index=False, encoding="utf-8")
print(f"Guardado CSV con PLS: {csv_out}")
print("Latencia promedio (s):", df_output["latency_s"].mean())


#3.7 a 5.8 GB VRAM


Generando resúmenes:   0%|          | 0/380 [00:00<?, ?sample/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tok

Guardado CSV con PLS: models\results\summaries_qwen3.csv
Latencia promedio (s): 39.99988362500076
